# Step 1: Loading the Dataset for Preprocessing

In this notebook, we prepare the Health Indicators dataset for machine learning models.

The preprocessing steps include removing unnecessary columns, encoding categorical variables, creating additional features, and preparing the dataset for regression and classification tasks.

The target variable is `stress_level`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path(r"C:\Users\onury\OneDrive\Masaüstü\Machine Learning\ML-Stress-Level-Prediction")

csv_path = PROJECT_DIR / "data" / "raw" / "health_indicators.csv"

df = pd.read_csv(csv_path)

df.head()

,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,stress_level,workouts_per_week,diet_quality,hydration_liters,bmi,heart_rate_resting,heart_rate_avg,calories_burned,consistency_score,lifestyle_category,fitness_level
0,56,12633,122,1084,8.65,8.67,4.42,2,5.61,2.14,16.00,54.8,103.0,1500.0,3.84,Active,Moderate
1,46,1980,33,1122,5.97,5.37,7.91,2,8.17,0.92,37.07,75.2,126.3,1500.0,7.25,Active,Unfit
2,32,7139,60,1049,6.23,7.15,4.20,0,3.47,2.67,27.36,66.3,101.4,1500.0,5.77,Sedentary,Unfit
3,60,4814,66,1018,5.09,7.61,7.12,4,5.26,2.17,24.59,64.6,107.8,1500.0,9.11,Active,Moderate
4,25,8096,86,1031,5.49,5.57,3.88,3,2.60,2.91,19.07,64.0,92.8,1500.0,7.55,Active,Moderate


In [3]:
print("Dataset Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

Dataset Shape: (1000000, 17)
Missing Values: 0
Duplicate Rows: 0


# Step 2: Removing Constant Features

During exploratory data analysis, the `calories_burned` column was found to contain only one unique value.

Since this feature is constant for all rows, it does not provide useful information for prediction.

Therefore, it will be removed before model training.

In [4]:
df["calories_burned"].nunique()

1

In [5]:
df = df.drop(columns=["calories_burned"])

df.head()

,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,stress_level,workouts_per_week,diet_quality,hydration_liters,bmi,heart_rate_resting,heart_rate_avg,consistency_score,lifestyle_category,fitness_level
0,56,12633,122,1084,8.65,8.67,4.42,2,5.61,2.14,16.00,54.8,103.0,3.84,Active,Moderate
1,46,1980,33,1122,5.97,5.37,7.91,2,8.17,0.92,37.07,75.2,126.3,7.25,Active,Unfit
2,32,7139,60,1049,6.23,7.15,4.20,0,3.47,2.67,27.36,66.3,101.4,5.77,Sedentary,Unfit
3,60,4814,66,1018,5.09,7.61,7.12,4,5.26,2.17,24.59,64.6,107.8,9.11,Active,Moderate
4,25,8096,86,1031,5.49,5.57,3.88,3,2.60,2.91,19.07,64.0,92.8,7.55,Active,Moderate


In [6]:
df.shape

(1000000, 16)

# Step 3: Encoding Categorical Variables

The dataset contains two categorical variables: `lifestyle_category` and `fitness_level`.

Machine learning models require numerical input, so these categorical variables need to be converted into numerical format.

In this step, one-hot encoding will be applied to transform categorical columns into binary numerical columns.

In [7]:
categorical_columns = ["lifestyle_category", "fitness_level"]

df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

df_encoded.head()

,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,stress_level,workouts_per_week,diet_quality,hydration_liters,bmi,heart_rate_resting,heart_rate_avg,consistency_score,lifestyle_category_Athlete,lifestyle_category_Sedentary,fitness_level_Moderate,fitness_level_Unfit
0,56,12633,122,1084,8.65,8.67,4.42,2,5.61,2.14,16.00,54.8,103.0,3.84,False,False,True,False
1,46,1980,33,1122,5.97,5.37,7.91,2,8.17,0.92,37.07,75.2,126.3,7.25,False,False,False,True
2,32,7139,60,1049,6.23,7.15,4.20,0,3.47,2.67,27.36,66.3,101.4,5.77,False,True,False,True
3,60,4814,66,1018,5.09,7.61,7.12,4,5.26,2.17,24.59,64.6,107.8,9.11,False,False,True,False
4,25,8096,86,1031,5.49,5.57,3.88,3,2.60,2.91,19.07,64.0,92.8,7.55,False,False,True,False


## Interpretation of Categorical Encoding

The categorical columns `lifestyle_category` and `fitness_level` were converted into numerical format using one-hot encoding.

After encoding, new binary columns were created:

- `lifestyle_category_Athlete`
- `lifestyle_category_Sedentary`
- `fitness_level_Moderate`
- `fitness_level_Unfit`

The original categorical columns were removed, and the dataset now contains only numerical features.

After this step, the dataset shape became 1,000,000 rows and 18 columns.

In [8]:
df_encoded.shape

(1000000, 18)

In [9]:
df_encoded.columns

Index(['age', 'steps_per_day', 'active_minutes', 'sedentary_minutes',
       'sleep_hours', 'sleep_quality', 'stress_level', 'workouts_per_week',
       'diet_quality', 'hydration_liters', 'bmi', 'heart_rate_resting',
       'heart_rate_avg', 'consistency_score', 'lifestyle_category_Athlete',
       'lifestyle_category_Sedentary', 'fitness_level_Moderate',
       'fitness_level_Unfit'],
      dtype='str')

# Step 4: Feature Engineering

In this step, new features are created from existing columns to provide additional useful information for machine learning models.

The target variable `stress_level` is not used during feature engineering to avoid data leakage.

The following new features are created:

- `activity_ratio`: ratio of active minutes to total active and sedentary minutes
- `sleep_deficit`: absolute difference between 8 hours and actual sleep duration
- `heart_rate_difference`: difference between average heart rate and resting heart rate

In [10]:
df_processed = df_encoded.copy()

df_processed["activity_ratio"] = df_processed["active_minutes"] / (
    df_processed["active_minutes"] + df_processed["sedentary_minutes"]
)

df_processed["sleep_deficit"] = abs(8 - df_processed["sleep_hours"])

df_processed["heart_rate_difference"] = (
    df_processed["heart_rate_avg"] - df_processed["heart_rate_resting"]
)

df_processed.head()

,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,stress_level,workouts_per_week,diet_quality,hydration_liters,...,heart_rate_resting,heart_rate_avg,consistency_score,lifestyle_category_Athlete,lifestyle_category_Sedentary,fitness_level_Moderate,fitness_level_Unfit,activity_ratio,sleep_deficit,heart_rate_difference
0,56,12633,122,1084,8.65,8.67,4.42,2,5.61,2.14,...,54.8,103.0,3.84,False,False,True,False,0.101161,0.65,48.2
1,46,1980,33,1122,5.97,5.37,7.91,2,8.17,0.92,...,75.2,126.3,7.25,False,False,False,True,0.028571,2.03,51.1
2,32,7139,60,1049,6.23,7.15,4.20,0,3.47,2.67,...,66.3,101.4,5.77,False,True,False,True,0.054103,1.77,35.1
3,60,4814,66,1018,5.09,7.61,7.12,4,5.26,2.17,...,64.6,107.8,9.11,False,False,True,False,0.060886,2.91,43.2
4,25,8096,86,1031,5.49,5.57,3.88,3,2.60,2.91,...,64.0,92.8,7.55,False,False,True,False,0.076992,2.51,28.8


In [11]:
df_processed.shape

(1000000, 21)

In [12]:
df_processed[["activity_ratio", "sleep_deficit", "heart_rate_difference"]].describe()

,activity_ratio,sleep_deficit,heart_rate_difference
count,1000000.000000,1000000.000000,1000000.000000
mean,0.061423,1.619394,40.035577
std,0.025369,1.028227,9.894803
min,0.008264,0.000000,3.300000
25%,0.043739,0.790000,33.200000
50%,0.061044,1.520000,40.000000
75%,0.078469,2.310000,46.700000
max,0.181628,5.000000,89.500000


## Interpretation of Engineered Features

Three new features were created during feature engineering: `activity_ratio`, `sleep_deficit`, and `heart_rate_difference`.

The `activity_ratio` feature represents the proportion of active minutes compared to total active and sedentary minutes.

The `sleep_deficit` feature represents how far a person's sleep duration is from the ideal 8 hours of sleep.

The `heart_rate_difference` feature represents the difference between average heart rate and resting heart rate.

The target variable `stress_level` was not used while creating these features in order to avoid data leakage.

After feature engineering, the dataset contains 1,000,000 rows and 21 columns.

# Step 5: Preparing Data for Regression

In this step, the dataset is prepared for the regression task.

For regression, the target variable is `stress_level`, which is a numerical value between 1 and 10.

The input features are all columns except `stress_level`.

This setup allows machine learning models to predict the numerical stress level from health and lifestyle indicators.

In [13]:
X_regression = df_processed.drop(columns=["stress_level"])
y_regression = df_processed["stress_level"]

print("X shape:", X_regression.shape)
print("y shape:", y_regression.shape)

X shape: (1000000, 20)
y shape: (1000000,)


In [14]:
"stress_level" in X_regression.columns

False

In [16]:
X_regression.head()

,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,workouts_per_week,diet_quality,hydration_liters,bmi,heart_rate_resting,heart_rate_avg,consistency_score,lifestyle_category_Athlete,lifestyle_category_Sedentary,fitness_level_Moderate,fitness_level_Unfit,activity_ratio,sleep_deficit,heart_rate_difference
0,56,12633,122,1084,8.65,8.67,2,5.61,2.14,16.00,54.8,103.0,3.84,False,False,True,False,0.101161,0.65,48.2
1,46,1980,33,1122,5.97,5.37,2,8.17,0.92,37.07,75.2,126.3,7.25,False,False,False,True,0.028571,2.03,51.1
2,32,7139,60,1049,6.23,7.15,0,3.47,2.67,27.36,66.3,101.4,5.77,False,True,False,True,0.054103,1.77,35.1
3,60,4814,66,1018,5.09,7.61,4,5.26,2.17,24.59,64.6,107.8,9.11,False,False,True,False,0.060886,2.91,43.2
4,25,8096,86,1031,5.49,5.57,3,2.60,2.91,19.07,64.0,92.8,7.55,False,False,True,False,0.076992,2.51,28.8


In [17]:
y_regression.head()

0    4.42
1    7.91
2    4.20
3    7.12
4    3.88
Name: stress_level, dtype: float64

# Step 6: Train-Test Split for Regression

In this step, the regression dataset is divided into training and testing sets.

The training set is used to train the machine learning models, while the testing set is used to evaluate model performance on unseen data.

We use 80% of the data for training and 20% for testing.

In [19]:
import sys
!{sys.executable} -m pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 5.0 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/8.1 MB 5.5 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/8.1 MB 7.0 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/8.1 MB 7.4 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.1 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 7.6 MB/s  0:00:01
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.3 MB 9.2 MB/s eta 0:00:04
   --- ------------------------------------ 3.4/37.3 MB 8.5 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/37.3 MB 7.4 MB/s eta 0:00:05
   ------ --------------------------------- 6.0/37.3 MB 7.4 MB/s eta 0:00:05
   -------- ------------------------------- 8.1/37.3 MB 8.0 MB/s eta 0:00:04
   ---------- ---------

In [20]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_regression,
    y_regression,
    test_size=0.2,
    random_state=42
)

print("X_train_reg shape:", X_train_reg.shape)
print("X_test_reg shape:", X_test_reg.shape)
print("y_train_reg shape:", y_train_reg.shape)
print("y_test_reg shape:", y_test_reg.shape)

X_train_reg shape: (800000, 20)
X_test_reg shape: (200000, 20)
y_train_reg shape: (800000,)
y_test_reg shape: (200000,)


## Interpretation of Regression Train-Test Split

The regression dataset was split into training and testing sets.

80% of the data was assigned to the training set, and 20% was assigned to the testing set.

The training set contains 800,000 records, while the testing set contains 200,000 records.

This split will be used later to train and evaluate regression models for predicting the numerical `stress_level` value.

# Step 7: Preparing Data for Classification

In this step, the numerical `stress_level` variable is converted into categorical stress classes.

The stress levels are divided into three categories:

- Low Stress: stress level below 4
- Medium Stress: stress level from 4 to below 7
- High Stress: stress level from 7 to 10

This allows the project to be handled as a classification problem in addition to regression.

In [21]:
def categorize_stress(value):
    if value < 4:
        return "Low"
    elif value < 7:
        return "Medium"
    else:
        return "High"


df_processed["stress_category"] = df_processed["stress_level"].apply(categorize_stress)

df_processed[["stress_level", "stress_category"]].head()

,stress_level,stress_category
0,4.42,Medium
1,7.91,High
2,4.20,Medium
3,7.12,High
4,3.88,Low


In [22]:
df_processed["stress_category"].value_counts()

stress_category
Medium    551876
High      318985
Low       129139
Name: count, dtype: int64

In [23]:
df_processed["stress_category"].value_counts(normalize=True) * 100

stress_category
Medium    55.1876
High      31.8985
Low       12.9139
Name: proportion, dtype: float64

## Interpretation of Stress Category Distribution

The numerical `stress_level` variable was converted into three categorical classes: Low, Medium, and High.

The largest class is Medium stress, which represents approximately 55.19% of the dataset. The High stress class represents approximately 31.90%, while the Low stress class represents approximately 12.91%.

This shows that the classification target is not perfectly balanced. However, all three classes have a sufficient number of records for model training.

Because the class distribution is imbalanced, stratified train-test splitting will be used to preserve the same class proportions in both training and testing sets.

# Step 8: Preparing Data for Classification

In this step, the input features and target variable are prepared for the classification task.

The target variable is `stress_category`.

Both `stress_level` and `stress_category` are removed from the input features to avoid data leakage.

The model will try to predict whether a person belongs to the Low, Medium, or High stress category using health and lifestyle indicators.

In [24]:
X_classification = df_processed.drop(columns=["stress_level", "stress_category"])
y_classification = df_processed["stress_category"]

print("X_classification shape:", X_classification.shape)
print("y_classification shape:", y_classification.shape)

X_classification shape: (1000000, 20)
y_classification shape: (1000000,)


In [25]:
print("stress_level in X_classification:", "stress_level" in X_classification.columns)
print("stress_category in X_classification:", "stress_category" in X_classification.columns)

stress_level in X_classification: False
stress_category in X_classification: False


# Step 9: Train-Test Split for Classification

In this step, the classification dataset is divided into training and testing sets.

Since the stress category distribution is not perfectly balanced, stratified splitting is used.

Stratified splitting preserves the class proportions of Low, Medium, and High stress in both the training and testing sets.

In [26]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_classification,
    y_classification,
    test_size=0.2,
    random_state=42,
    stratify=y_classification
)

print("X_train_cls shape:", X_train_cls.shape)
print("X_test_cls shape:", X_test_cls.shape)
print("y_train_cls shape:", y_train_cls.shape)
print("y_test_cls shape:", y_test_cls.shape)

X_train_cls shape: (800000, 20)
X_test_cls shape: (200000, 20)
y_train_cls shape: (800000,)
y_test_cls shape: (200000,)


In [27]:
print("Training target distribution:")
print(y_train_cls.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test_cls.value_counts(normalize=True) * 100)

Training target distribution:
stress_category
Medium    55.187625
High      31.898500
Low       12.913875
Name: proportion, dtype: float64

Testing target distribution:
stress_category
Medium    55.1875
High      31.8985
Low       12.9140
Name: proportion, dtype: float64


## Interpretation of Classification Train-Test Split

The classification dataset was split into training and testing sets using an 80%-20% ratio.

Because the stress category distribution is not perfectly balanced, stratified splitting was applied. This preserved the proportions of Low, Medium, and High stress categories in both the training and testing sets.

The training set contains 800,000 records, and the testing set contains 200,000 records.

The class distributions in the training and testing sets are almost identical, which confirms that the stratified split was successful.

# Step 10: Preprocessing Summary

In this notebook, the Health Indicators dataset was prepared for machine learning models.

First, the dataset was loaded and checked. The dataset contains 1,000,000 rows and initially had 17 columns.

The `calories_burned` column was removed because it had only one unique value and did not provide useful information for prediction.

The categorical columns `lifestyle_category` and `fitness_level` were converted into numerical format using one-hot encoding.

Three new features were created:

- `activity_ratio`
- `sleep_deficit`
- `heart_rate_difference`

For the regression task, the target variable is the numerical `stress_level` column.

For the classification task, the numerical `stress_level` variable was converted into three categories: Low, Medium, and High.

Both regression and classification datasets were split into training and testing sets using an 80%-20% ratio.

For classification, stratified splitting was used to preserve the class distribution in both training and testing sets.

The dataset is now ready for model training.

In [28]:
print("Final processed dataset shape:", df_processed.shape)

print("\nRegression:")
print("X_train_reg:", X_train_reg.shape)
print("X_test_reg:", X_test_reg.shape)
print("y_train_reg:", y_train_reg.shape)
print("y_test_reg:", y_test_reg.shape)

print("\nClassification:")
print("X_train_cls:", X_train_cls.shape)
print("X_test_cls:", X_test_cls.shape)
print("y_train_cls:", y_train_cls.shape)
print("y_test_cls:", y_test_cls.shape)

Final processed dataset shape: (1000000, 22)

Regression:
X_train_reg: (800000, 20)
X_test_reg: (200000, 20)
y_train_reg: (800000,)
y_test_reg: (200000,)

Classification:
X_train_cls: (800000, 20)
X_test_cls: (200000, 20)
y_train_cls: (800000,)
y_test_cls: (200000,)


# Step 11: Saving the Processed Dataset

The processed dataset is saved as a CSV file so that it can be reused in the model training notebooks.

In [29]:
processed_data_path = PROJECT_DIR / "data" / "processed"
processed_data_path.mkdir(parents=True, exist_ok=True)

df_processed.to_csv(processed_data_path / "processed_health_indicators.csv", index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
